## imports

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn

sys.path.append('../src')
from load_clean_data import load_clean_data

In [2]:
df = load_clean_data()
print(df.shape)

c:\Users\corbi\Documents\Projects\sales-forecasting\sales-forecasting-project\notebooks\../src\load_clean_data.py:6: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../data/train.csv')


(1017209, 18)


## naive baseline (7-day lag)

### data integrity check

Before building a 7-day lag feature for the naive baseline, checking whether every store has a complete daily record with no gaps. a gap would make a simple row-based shift unsafe bypulling sales from the wrong date

In [3]:
store1 = df[df['Store'] == 1]

row_count = len(store1)
day_count = (store1['Date'].max() - store1['Date'].min()).days + 1

print(row_count, day_count)

942 942


In [4]:
row_counts = df.groupby('Store')['Date'].count()
day_counts = df.groupby('Store')['Date'].agg(lambda x: (x.max() - x.min()).days + 1)

mismatches = row_counts != day_counts
mismatches.sum()

np.int64(180)

Finding: 180 of 1,115 stores (about 16%) have at least one gap in their daily records. Investigating one example to understand what a gap actually looks like.

In [5]:
gap_stores = row_counts[mismatches].index
example_store = df[df['Store'] == gap_stores[0]]
example_store[['Date', 'Open']].sort_values('Date')

,Date,Open
1016107,2013-01-01,0
1014992,2013-01-02,1
1013877,2013-01-03,1
1012762,2013-01-04,1
1011647,2013-01-05,1
...,...,...
4472,2015-07-27,1
3357,2015-07-28,1
2242,2015-07-29,1
1127,2015-07-30,1


In [6]:
example_store_sorted = example_store.sort_values('Date')
example_store_sorted['date_diff'] = example_store_sorted['Date'].diff()
example_store_sorted['date_diff'].value_counts()

date_diff
1 days      756
185 days      1
Name: count, dtype: int64

Finding: this store has a single clean 185-day gap, not scattered small gaps. Likely a store that was closed or untracked for an extended period — the exact cause doesn't change how it needs to be handled, so not investigating further.

### data preparation

A plain .shift(7) would be unsafe here, since it shifts by row position, not by actual date. Near a gap it would pull sales from the wrong day. Instead, building a lookup table that shifts the Date itself forward by 7 days, then merging back onto the original data by Store and Date. This way, missing dates correctly produce NaN instead of a wrong value.

In [7]:
lag_lookup = df[['Store', 'Date', 'Sales']].copy()

lag_lookup['Date'] = lag_lookup['Date'] + pd.Timedelta(days=7)
lag_lookup = lag_lookup.rename(columns={'Sales': 'Sales_lag7'})

df = df.merge(lag_lookup, on=['Store', 'Date'], how='left')

check = df[df['Store'] == gap_stores[0]].sort_values('Date')
check[['Date', 'Sales', 'Sales_lag7']].head(20)

,Date,Sales,Sales_lag7
1016107,2013-01-01,0,NaN
1014992,2013-01-02,3737,NaN
1013877,2013-01-03,3674,NaN
1012762,2013-01-04,3848,NaN
1011647,2013-01-05,4285,NaN
1010532,2013-01-06,0,NaN
1009417,2013-01-07,8246,NaN
1008302,2013-01-08,6700,0.0
1007187,2013-01-09,5296,3737.0
1006072,2013-01-10,5432,3674.0


In [8]:
example_store_sorted[example_store_sorted['date_diff'] == pd.Timedelta(days=185)]

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,date_diff
235277,13,4,2015-01-01,0,0,0,0,a,1,d,a,310.0,NaN,NaN,1,45.0,2009.0,"Feb,May,Aug,Nov",185 days


So one of the gaps we found earlier, starts at 2015-01-01 for this store

In [9]:
store13 = df[df['Store'] == 13].sort_values('Date')
store13[store13['Date'] >= '2014-12-25'][['Date', 'Sales', 'Sales_lag7']].head(15)

,Date,Sales,Sales_lag7
235277,2015-01-01,0,NaN
234162,2015-01-02,4348,NaN
233047,2015-01-03,4066,NaN
231932,2015-01-04,0,NaN
230817,2015-01-05,10460,NaN
229702,2015-01-06,0,NaN
228587,2015-01-07,6988,NaN
227472,2015-01-08,6405,0.0
226357,2015-01-09,6241,4348.0
225242,2015-01-10,4315,4066.0


Verified: Sales_lag7 correctly shows NaN both for the first week of the dataset (no prior data exists) and for the week immediately after a real 185-day gap in Store 13's data (checked directly). Elsewhere, values correctly match real Sales from exactly 7 days earlier (e.g., Jan 9 2015's lag value of 4348 matches Jan 2's actual Sales). 

TODO: the gaps themselves are still present in the raw data — revisit whether later models (SARIMA, ML) need explicit handling for them.

## seasonal average baseline

Predicting a store's sales on a given day as the average of that store's past sales on the same day-of-week (e.g. Store 13's typical Monday). Using an expanding average, shifted by one, so each prediction only uses prior occurrences of that day-of-week — never the day itself or anything in the future. This avoids data leakage.

In [12]:
df = df.sort_values(['Store', 'DayOfWeek', 'Date'])

seasonal_avg = (
    df.groupby(['Store', 'DayOfWeek'])['Sales']
      .expanding()
      .mean()
      .shift(1)
)

seasonal_avg.index = seasonal_avg.index.droplevel([0, 1])
df['Sales_seasonal_avg'] = seasonal_avg

First attempt misaligned the result with the wrong rows after a .reset_index(drop=True) — groupby output isn't in the same row order as the original dataframe, so resetting the index blindly scrambled the values. Fixed by dropping only the groupby's extra index levels and letting pandas align by the original row index instead.

In [13]:
example = df[(df['Store']== 1) & (df['DayOfWeek'] == 1)]


example = example.sort_values('Date')
example[['Date', 'Sales', 'Sales_seasonal_avg']].head(10)

,Date,Sales,Sales_seasonal_avg
1009405,2013-01-07,7176,NaN
1001600,2013-01-14,4717,7176.000000
993795,2013-01-21,5394,5946.500000
985990,2013-01-28,4055,5762.333333
978185,2013-02-04,7032,5335.500000
970380,2013-02-11,4409,5674.800000
962575,2013-02-18,6407,5463.833333
954770,2013-02-25,4038,5598.571429
946965,2013-03-04,7675,5403.500000
939160,2013-03-11,4949,5655.888889


Verified: for Store 1's Mondays, the first row correctly shows NaN (no prior Monday exists yet), and each subsequent row's average matches the manual calculation from prior Mondays only (e.g. row 2 = row 1's Sales exactly; row 3 = average of rows 1–2). No data leakage.

## evaluation

using RMSE and MAPE

In [ ]:
naive_valid = df[['Sales', 'Sales_lag7']].dropna()
seasonal_avg_valid = df[['Sales', 'Sales_seasonal_avg']].dropna()

naive_RMSE = sklearn.metrics.root_mean_squared_error(naive_valid['Sales'], naive_valid['Sales_lag7'])
seasonal_avg_RMSE = sklearn.metrics.root_mean_squared_error(seasonal_avg_valid['Sales'], seasonal_avg_valid['Sales_seasonal_avg'])

naive_MAE = sklearn.metrics.mean_absolute_error(naive_valid['Sales'], naive_valid['Sales_lag7'])
seasonal_avg_MAE = sklearn.metrics.mean_absolute_error(seasonal_avg_valid['Sales'], seasonal_avg_valid['Sales_seasonal_avg'])

print(f"Naive RMSE: {naive_RMSE:.2f}, Seasonal Avg RMSE: {seasonal_avg_RMSE:.2f}")
print(f"Naive MAE: {naive_MAE:.2f}, Seasonal Avg MAE: {seasonal_avg_MAE:.2f}")

Naive RMSE: 3101.09, Seasonal Avg RMSE: 2101.13
Naive MAPE: 957951312375662080.0000, Seasonal Avg MAPE: 951791958365227008.0000
